In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# ==============================
# 설정값 (ipynb에서 경로만 바꿔 사용)
# ==============================
STYLE_PATH = 'imgs/grammatrix/vangogh_starry_night.jpg'
CONTENT_PATH = 'imgs/grammatrix/Tuebingen_Neckarfront.jpg'
IMG_SIZE = 512
NUM_ITERATIONS = 500

# ==============================
# Gram Matrix / Loss
# ==============================
class GramMatrix(nn.Module):
    def forward(self, input):
        b, c, h, w = input.size()
        feat = input.view(b, c, h * w)
        gram = torch.bmm(feat, feat.transpose(1, 2))
        gram.div_(h * w)
        return gram

class GramMSELoss(nn.Module):
    def forward(self, input, target):
        return nn.MSELoss()(GramMatrix()(input), target)

# ==============================
# VGG 구조 (원본 ipynb 그대로, pretrained 미사용)
# ==============================
class VGG(nn.Module):
    def __init__(self, pool='max'):
        super().__init__()
        self.conv1_1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv2_1 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, 3, padding=1)
        self.conv3_1 = nn.Conv2d(128, 256, 3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv3_4 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv4_1 = nn.Conv2d(256, 512, 3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv4_4 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_1 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_2 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_3 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_4 = nn.Conv2d(512, 512, 3, padding=1)
        Pool = nn.MaxPool2d if pool == 'max' else nn.AvgPool2d
        self.pool1 = Pool(2,2); self.pool2 = Pool(2,2); self.pool3 = Pool(2,2)
        self.pool4 = Pool(2,2); self.pool5 = Pool(2,2)

    def forward(self, x, out_keys):
        out = {}
        out['r11'] = F.relu(self.conv1_1(x))
        out['r12'] = F.relu(self.conv1_2(out['r11']))
        out['p1'] = self.pool1(out['r12'])
        out['r21'] = F.relu(self.conv2_1(out['p1']))
        out['r22'] = F.relu(self.conv2_2(out['r21']))
        out['p2'] = self.pool2(out['r22'])
        out['r31'] = F.relu(self.conv3_1(out['p2']))
        out['r32'] = F.relu(self.conv3_2(out['r31']))
        out['r33'] = F.relu(self.conv3_3(out['r32']))
        out['r34'] = F.relu(self.conv3_4(out['r33']))
        out['p3'] = self.pool3(out['r34'])
        out['r41'] = F.relu(self.conv4_1(out['p3']))
        out['r42'] = F.relu(self.conv4_2(out['r41']))
        out['r43'] = F.relu(self.conv4_3(out['r42']))
        out['r44'] = F.relu(self.conv4_4(out['r43']))
        out['p4'] = self.pool4(out['r44'])
        out['r51'] = F.relu(self.conv5_1(out['p4']))
        out['r52'] = F.relu(self.conv5_2(out['r51']))
        out['r53'] = F.relu(self.conv5_3(out['r52']))
        out['r54'] = F.relu(self.conv5_4(out['r53']))
        out['p5'] = self.pool5(out['r54'])
        return [out[k] for k in out_keys]

# ==============================
# 이미지 로드 / 전처리
# ==============================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

prep = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x[torch.LongTensor([2,1,0])]), # RGB -> BGR
    transforms.Normalize(mean=[0.40760392,0.45795686,0.48501961], std=[1,1,1]),
    transforms.Lambda(lambda x: x.mul_(255)),
])

style_image = prep(Image.open(STYLE_PATH)).unsqueeze(0).to(device)
content_image = prep(Image.open(CONTENT_PATH)).unsqueeze(0).to(device)
input_image = content_image.clone().requires_grad_(True)

# ==============================
# 모델 / 타깃 생성
# ==============================
vgg = VGG().to(device)
style_layers = ['r11','r21','r31','r41','r51']
content_layers = ['r42']
loss_layers = style_layers + content_layers

loss_fns = [GramMSELoss().to(device) for _ in style_layers] + [nn.MSELoss().to(device) for _ in content_layers]
style_weights = [1e3 / n**2 for n in [64,128,256,512,512]]
content_weights = [1e0]
weights = style_weights + content_weights

style_targets = [GramMatrix()(A).detach() for A in vgg(style_image, style_layers)]
content_targets = [A.detach() for A in vgg(content_image, content_layers)]
targets = style_targets + content_targets

# ==============================
# 최적화
# ==============================
optimizer = torch.optim.LBFGS([input_image], max_iter=1)
n_iter = 0

def closure():
    global n_iter
    optimizer.zero_grad()
    out = vgg(input_image, loss_layers)
    total_loss = 0
    for i, w in enumerate(weights):
        total_loss += w * loss_fns[i](out[i], targets[i])
    total_loss.backward()
    if n_iter % 50 == 0:
        print(f'Iteration {n_iter}: Total Loss={total_loss.item():.4f}')
    n_iter += 1
    return total_loss

for _ in range(NUM_ITERATIONS):
    optimizer.step(closure)

# ==============================
# 결과 표시 (ipynb용)
# ==============================
def show_result(tensor):
    img = tensor.detach().cpu().clone().squeeze(0)
    img = img / 255.0
    img = img[[2,1,0], :, :]
    img = torch.clamp(img, 0, 1)
    pil = transforms.ToPILImage()(img)
    plt.figure(figsize=(8,8))
    plt.imshow(pil)
    plt.axis('off')
    plt.title('Finalized_Image')
    plt.show()

show_result(input_image)
